# Tripwire: train the tool-output injection classifiers (Colab / Kaggle T4)

Trains **MiniLM-L6** (default candidate) and **DeBERTa-v3-small** (accuracy reference) on the same dataset and splits (seed 42), exports int8 ONNX artifacts in the gateway's format, evaluates them exactly as the gateway runs them, and optionally evaluates **Llama Prompt Guard 2 22M** zero-shot.

1. Runtime → Change runtime type → **T4 GPU** (Kaggle: Accelerator → GPU T4).
2. Run all cells. Rough time on a free T4 session: dataset build about 5 min, MiniLM selection and training about 15 min, DeBERTa-v3-small training about 35 min. Evaluation runs the int8 ONNX artifacts on CPU one window at a time, exactly as the gateway does: about 5 min for MiniLM and 30–40 min for DeBERTa. In total, about 1.5–2 hours.
3. Download `tripwire-artifacts.zip` at the end and unpack it into `ml/artifacts/`.

Prompt Guard 2 is gated (Llama 4 Community License): request access on its Hugging Face page and add an `HF_TOKEN` secret (Colab: key icon; Kaggle: Add-ons → Secrets). Its weights are evaluated locally and never published.

In [ ]:
!nvidia-smi -L || echo "no GPU: training will be slow"
!git clone --depth 1 https://github.com/yugpatill/tripwire-agent-gateway.git
%cd tripwire-agent-gateway/ml

In [ ]:
# Colab/Kaggle ship a CUDA build of torch; install the rest within the ml/ pins.
!pip -q install "transformers>=4.46,<5" "datasets>=3.1" "accelerate>=1.1" "scikit-learn>=1.5" \
    "optimum[onnxruntime]" "onnx>=1.17" evaluate matplotlib sentencepiece
import torch, transformers
print(torch.__version__, transformers.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
EVAL_PROMPT_GUARD = True                       # zero-shot baseline, needs HF_TOKEN + approved access

import os
try:
    from google.colab import userdata  # Colab
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
!python scripts/build_dataset.py

### Pick the MiniLM-L6 checkpoint
Both candidates share the same 6-layer, 384-wide architecture and 512 positions, so latency is identical:
* `nreimers/MiniLM-L6-H384-uncased` (MIT): a general-purpose distilled language model, the usual starting point for classification fine-tuning.
* `sentence-transformers/all-MiniLM-L6-v2` (Apache-2.0): the same backbone further tuned for sentence embeddings, on inputs of up to 256 tokens.

One epoch of each on the same data; the higher validation PR-AUC wins and is trained fully below.

In [ ]:
import json, pathlib
CANDIDATES = {"minilm-l6": "nreimers/MiniLM-L6-H384-uncased", "minilm-l6-st": "sentence-transformers/all-MiniLM-L6-v2"}
selection = {}
for key, hf_id in CANDIDATES.items():
    !python scripts/train.py --model {key} --epochs 1 --output-key select-{key}
    info = json.loads(pathlib.Path(f"checkpoints/select-{key}/best/train_info.json").read_text())
    selection[hf_id] = info["val"]
best_hf = max(selection, key=lambda k: selection[k]["eval_pr_auc"])
print(json.dumps(selection, indent=2)); print("selected:", best_hf)
pathlib.Path("artifacts").mkdir(exist_ok=True)
pathlib.Path("artifacts/minilm_selection.json").write_text(json.dumps({"candidates": selection, "selected": best_hf}, indent=2))
!rm -rf checkpoints/select-*

In [ ]:
!python scripts/train.py --model minilm-l6 --hf-id {best_hf}
!python scripts/export_onnx.py --model minilm-l6
!python scripts/evaluate.py --model minilm-l6 --threads 2

!python scripts/train.py --model deberta-v3-small
!python scripts/export_onnx.py --model deberta-v3-small
!python scripts/evaluate.py --model deberta-v3-small --threads 2

In [ ]:
if EVAL_PROMPT_GUARD and os.environ.get("HF_TOKEN"):
    !python scripts/export_onnx.py --model prompt-guard-2-22m
    !python scripts/evaluate.py --model prompt-guard-2-22m --threads 2
else:
    print("skipping Prompt Guard 2 (no HF_TOKEN)")

In [ ]:
import json, pathlib
for d in sorted(pathlib.Path("artifacts").glob("injection-*")):
    m = json.loads((d / "metrics.json").read_text())
    t, o = m["test"], m["test_ood"]
    print(f"{d.name:32} thr={m['threshold']:.4f} test R={t['recall']:.3f} P={t['precision']:.3f} "
          f"AUC={t['roc_auc']:.3f} | ood R={o['recall']:.3f} | hardneg FPR={t['fpr_hard_negatives']:.3f}")

In [ ]:
# Package the publishable artifacts (Prompt Guard 2 is excluded: license, evaluation only).
!cd artifacts && zip -qr ../tripwire-artifacts.zip injection-minilm-l6 injection-deberta-v3-small minilm_selection.json \
    && for d in injection-prompt-guard-2-22m; do [ -d $d ] && zip -qr ../tripwire-artifacts.zip $d/metrics.json $d/pr_curve.png; done; true
try:
    from google.colab import files
    files.download("tripwire-artifacts.zip")
except Exception:
    print("Download tripwire-artifacts.zip from the file browser (Kaggle: /kaggle/working).")